# 📌 **Etapa 1 - Instalação das Ferramentas e Configuração Inicial**
# **Objetivo**

Garantir que o conjunto de dados esteja estruturado e disponível no ambiente, preparando as bibliotecas necessárias.

Nesta fase são realizadas:
*  Instalação das bibliotecas;
*  Configuração para garantir reprodutibilidade;
*  Definição dos domínios realistas para as URLs por tipo de fonte.




In [2]:
# Instalandos as bibliotecas necessárias
!pip install pandas numpy scikit-learn faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 12.9 MB/s eta 0:00:00


In [3]:
import random
import math
import unicodedata
import re
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
from sklearn.feature_extraction.text import TfidfVectorizer

# Configuração da Semente Obrigatória (Reprodutibilidade)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
fake = Faker('pt_BR')
Faker.seed(RANDOM_SEED)

print("Iniciando a preparação do ambiente")

# Mapeamento de domínios realistas por tipo de fonte
dominios_por_tipo = {
    'blog_tecnico': 'https://devblog.com.br/post/',
    'portal_educacional': 'https://educabytes.com.br/artigos/',
    'forum': 'https://comunidadedev.com.br/topico/',
    'tutorial': 'https://guiaprogramacao.com.br/tutoriais/',
    'site_institucional': 'https://techinstitucional.com.br/docs/'
}

Iniciando a preparação do ambiente


#**📌 Etapa 2 - Construção Programática dos Conteúdos e Metadados**
#**Objetivo**

Gerar de forma programática os artigos e metadados respeitando estritamente a proporção temática e o tamanho exigido de palavras.

Nesta fase são realizadas:

* Criação de blocos de conteúdo sobre tecnologia e conhecimentos gerais;

* Distribuição controlada dos tamanhos de texto (curto, médio, longo e extenso);

* Geração de metadados como autores, datas, categorias e URLs.

In [4]:
# Blocos de conhecimento factuais para montagem
blocos_tech = [
    ("O desenvolvimento de APIs RESTful exige atenção rigorosa aos verbos HTTP e aos códigos de status.", "Uma boa modelagem de dados evita gargalos de performance quando o sistema passa por escalabilidade horizontal.", "O uso de tokens JWT permite autenticação stateless, reduzindo a carga contínua de consultas ao banco de dados.", "Em arquiteturas de microsserviços, a observabilidade através de logs estruturados e métricas consolidadas é indispensável.", "A refatoração de código seguindo os princípios SOLID facilita a manutenção e a criação de testes unitários robustos."),
    ("Bancos de dados relacionais utilizam normalização para evitar redundância, enquanto bancos NoSQL priorizam a flexibilidade do schema.", "O uso de índices em colunas frequentemente buscadas acelera consultas SQL, mas pode prejudicar o tempo de operações de escrita.", "A conteinerização com Docker isola as dependências da aplicação, garantindo paridade total entre ambientes de desenvolvimento e produção.", "Pipelines de CI/CD automatizam a execução de testes e o deploy, reduzindo falhas humanas na entrega de software.", "A segurança em aplicações web exige proteção ativa contra injeção de SQL e ataques de Cross-Site Scripting (XSS).")
]

blocos_gerais = [
    ("A Revolução Industrial transformou as relações de trabalho e impulsionou o crescimento acelerado dos centros urbanos no século XIX.", "Na fotografia profissional, o triângulo de exposição equilibra a abertura do diafragma, a velocidade do obturador e a sensibilidade ISO.", "O estudo da filosofia antiga permite compreender as bases da ética, da lógica e do pensamento científico ocidental.", "A preservação da biodiversidade e a gestão sustentável dos recursos hídricos são desafios centrais para a economia contemporânea.", "O cinema, como arte e meio de comunicação, reflete as tensões sociais de sua época através de elementos narrativos e visuais."),
    ("A literatura clássica oferece investigações profundas sobre a condição humana, atravessando fronteiras culturais e temporais.", "O método científico baseia-se na observação empírica, na formulação de hipóteses testáveis e na revisão pelos pares.", "A transição para fontes de energia renovável, como a solar e a eólica, é fundamental para mitigar as mudanças climáticas globais.", "A prática de exercícios físicos regulares e a alimentação balanceada estão diretamente ligadas à neuroplasticidade e ao bem-estar mental.", "A sociologia investiga como as estruturas de poder e as instituições sociais moldam o comportamento dos indivíduos na coletividade.")
]

def gerar_texto_realista(eh_tech, meta_palavras):
    blocos = blocos_tech if eh_tech else blocos_gerais
    paragrafos = []
    palavras_atuais = 0
    while palavras_atuais < meta_palavras:
        grupo = random.choice(blocos)
        frase = random.choice(grupo)
        paragrafos.append(frase)
        palavras_atuais += len(frase.split())
    texto = " ".join(paragrafos)
    palavras = texto.split()
    if len(palavras) > 783:
        texto = " ".join(palavras[:random.randint(551, 780)])
    return texto

dados = []
autores_pool = [fake.name() for _ in range(600)]
faixas_palavras = (
    [(random.randint(121, 220), 'curto')] * 1750 +
    [(random.randint(221, 380), 'medio')] * 2000 +
    [(random.randint(381, 550), 'longo')] * 1000 +
    [(random.randint(551, 783), 'extenso')] * 250
)
random.shuffle(faixas_palavras)

print("Gerando os registros")
for i in range(1, 5001):
    eh_tech = i <= 3500
    meta_palavras, content_size = faixas_palavras[i-1]

    content = gerar_texto_realista(eh_tech, meta_palavras)
    word_count = len(content.split())
    reading_time = max(1, math.ceil(word_count / 200))

    created_at = fake.date_time_between(start_date=datetime(2018,1,1), end_date=datetime(2026,6,30))
    updated_at = created_at + timedelta(days=random.randint(0, 100)) if random.random() > 0.08 else None

    source_type = random.choice(list(dominios_por_tipo.keys()))
    base_url = dominios_por_tipo[source_type]
    source_url = f"{base_url}{i}"

    categoria = random.choice(["Arquitetura", "Banco de Dados", "DevOps", "Backend", "Frontend", "Segurança"]) if eh_tech else random.choice(["História", "Ciência", "Artes", "Filosofia", "Meio Ambiente", "Sociologia"])
    titulo = f"{'Guia Prático' if eh_tech else 'Introdução ao Estudo'}: Conceitos de {categoria} - Parte {random.randint(1, 100)}"

    dados.append({
        "record_id": i,
        "source_id": f"SRC-{random.randint(1000, 9999)}",
        "source_type": source_type,
        "author_id": f"AUT-{random.randint(1, 600):04d}",
        "author": random.choice(autores_pool),
        "created_at": created_at.strftime("%Y-%m-%d %H:%M:%S"),
        "updated_at": updated_at.strftime("%Y-%m-%d %H:%M:%S") if updated_at else None,
        "title": titulo,
        "content": content,
        "category": categoria,
        "difficulty": random.choices(["Iniciante", "Intermediário", "Avançado"], weights=[0.45, 0.35, 0.20])[0],
        "language": "pt-BR",
        "content_size": content_size,
        "word_count": word_count,
        "reading_time_minutes": reading_time,
        "source_url": source_url,
        "duplicate_group_id": None,
        "quality_issue_count": 0,
        "quality_issues": "[]",
        "dataset_batch_id": "maria"
    })

df = pd.DataFrame(dados)

Gerando os registros


#**📌 Etapa 3 - Limpeza, Normalização Textual e Extração de Tags**
#**Objetivo**
Garantir que os textos estejam limpos e padronizados, enriquecendo o dataset com palavras-chave relevantes através de processamento estatístico.

Nesta fase são realizadas:

* Limpeza e Normalização: Remoção de acentos (NFKD), conversão para letras minúsculas e eliminação de caracteres especiais por meio de expressões regulares (re.sub), preparando o texto bruto para análise;

* Extração de Tags (TF-IDF): Aplicação do algoritmo estatístico para identificar os termos mais relevantes de cada artigo;

* Padronização do Resultado: Extração automática de 5 tags normalizadas por registro e formatação final do arquivo CSV.

In [7]:
print("Limpando, normalizando e extraindo tags")

def normalizar(t):
    return re.sub(r'[^a-z0-9\s]', '', unicodedata.normalize('NFKD', str(t)).encode('ASCII', 'ignore').decode('ASCII').lower())

vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2), min_df=2)
tfidf_mat = vectorizer.fit_transform(df['content'].apply(normalizar))
termos = np.array(vectorizer.get_feature_names_out())

tags_list = []
for i in range(len(df)):
    top_idx = tfidf_mat[i].toarray()[0].argsort()[-5:][::-1]
    t_ext = [termos[idx] for idx in top_idx if tfidf_mat[i].toarray()[0][idx] > 0]
    while len(t_ext) < 5:
        t_ext.append(random.choice(["tecnologia", "dados", "estudo", "conceito", "pratica"]))
    tags_list.append(str(t_ext[:5]).replace("'", '"'))
df['tags'] = tags_list

# Configurações de exibição e salvamento final
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

nome_arquivo = "synthetic_dataset.csv"
df.to_csv(nome_arquivo, index=False, encoding="utf-8")
print(f"O arquivo '{nome_arquivo}' limpo e estruturado")

Limpando, normalizando e extraindo tags
O arquivo 'synthetic_dataset.csv' limpo e estruturado
